
# Recipe Book: Building RM Task JSONs with `psychscanner`

This notebook is for someone who has never touched this codebase and wants to understand,
**from first principles**, how the task JSON files in `data/raw/rm_tasks/` are built and how
`psychscanner` turns them into a runnable simulation.

It is **not** one of the production generator notebooks (`make_rm_tasks_exp1.ipynb`,
`make_rm_tasks_exp2_nofb.ipynb`, `make_rm_tasks_exp2_fb.ipynb`) — those are the real thing,
terse and optimized for regenerating the full task set. This notebook instead builds small,
2-trial **toy versions** of each of the four distinct task "shapes" used in this project, step
by step, with an explanation at every field. Every toy example is cross-checked at the end
against the real file it corresponds to in `data/raw/rm_tasks/`, so you can trust that what
you're reading here is really how the production files are built, not a simplified fiction.

This notebook never writes into `data/raw/`. All output goes to `demo_output/`, a folder
created next to this notebook (`notebooks/01_task_gen/demo_output/`) the first time the setup
cell runs — separate from anything the real pipeline reads.

**The four task shapes covered:**

| # | Shape | `chain_type` | Real production file | What the LLM does |
|---|---|---|---|---|
| 1 | Single-turn, one stimulus per trial | `item` | `rm_2op.json` | Answers all 4 sub-questions about one word-pair in a single response |
| 2 | Single-turn, trial broken into steps | `trial` | `rm_2op_tc.json` | Same 4 sub-questions, but asked one at a time within the trial |
| 3 | Episodic conversation, no feedback | `task` | `rm_2op_convo.json` | Sees a word-pair, later gets tested on it in the same conversation |
| 4 | Episodic conversation, with feedback | `task` | `rm_2op_convo_feedback_allparts.json` | Same as #3, plus corrective feedback injected after each test response |

A fifth section covers the `_r` "reversed" counterbalancing trick used across all four shapes.


## Setup

Imports, and a scratch directory so this notebook never touches real project data.


In [ ]:
import copy
import json
from pathlib import Path

import rmllm
import psychscanner as psy

# Saved next to this notebook (notebooks/01_task_gen/demo_output/), not in data/raw/ --
# these are toy/teaching JSONs, not production task files.
SCRATCH_DIR = rmllm.config.PROJ_ROOT / "notebooks" / "01_task_gen" / "demo_output"
SCRATCH_DIR.mkdir(exist_ok=True)
print("Demo output directory:", SCRATCH_DIR)

RAW_DATA_DIR = rmllm.config.RAW_DATA_DIR
RM_TASKS_DIR = RAW_DATA_DIR / "rm_tasks"          # the real, production task files
PREPARE_DIR = RAW_DATA_DIR / "prepare_task"        # the raw stimuli + instruction-text inputs


## The blank slate: `psy.get_task_template()`

Every task JSON in this project starts from the same empty schema. `psychscanner` doesn't
require you to use this helper — a task JSON is "just" a dict matching this shape — but it's
the fastest way to see every field a task can have, with nothing filled in yet.


In [ ]:
blank_template = psy.get_task_template()
print(json.dumps(blank_template, indent=2))


**Field glossary** — what each key means and where it's actually used:

| Key | Meaning |
|---|---|
| `tasktype` | `"sc"` (single-chat, the only value used in this template) for one-shot/trial-chain tasks; the episodic conversation tasks override this to `"episodic_system"` directly (see shape 3/4). |
| `taskname` | A unique label. Ends up as a folder segment in the simulation output path, and as an identifier on every trial in the collected data — **must be unique per task file** (we verified this on the real files earlier). |
| `instructions` | The system/task instructions shown to the model. Structure depends on the task shape (see below). |
| `contexts` / `contexts_id` | The stimulus categories being manipulated — here always `["perceived", "imagined"]`, i.e. whether the second word in a pair was given to you or you had to imagine it. |
| `context_present` | Whether the context label itself is shown to the model as part of the stimulus (`False` in every file here — the model isn't told which condition a trial belongs to; that would give the answer away). |
| `items` | The actual trial data — one entry per trial, keyed by a `trcode` like `"imagined_1"`. |
| `chain_type` | `"item"` (one stimulus, one response), `"trial"` (one trial made of several sequential steps), or `"task"` (multi-turn conversation carrying memory across trials). This is the single biggest structural fork in the whole pipeline. |
| `trial_parsers` | A list of per-step parsers for `chain_type="trial"`. In every real file here it's left empty (`[]`) — the trial-level `parser` field below does the actual work instead. |
| `parser` | The name of a `pydantic` response-schema class (defined in `psychscanner.datasets.prompts.parser`) that the model's structured output must conform to. |
| `postfix` / `prefix` | Optional text glued before/after the stimulus. Unused (empty string) in every file here. |


## Recipe 1: Single-turn, item chain (`chain_type="item"`)

**Shape:** one word-pair stimulus per trial. The model is asked to do all four things in a
single response: report/imagine the second word, rate how related the two words are, judge
whether it generated `word_2` itself ("internal") or was given it ("external"), and rate its
confidence in that judgment. This is the simplest, most constrained task shape — no
conversation memory, no multi-step breakdown.

The production version, `rm_2op.json`, is built from two upstream files:
- `prepare_task/task_rm_v1.json` — the raw list of 144 word-pair stimuli.
- `prepare_task/rm_task_2op_inst.json` — the instruction text, with two option orderings
  (`order1`: external-then-internal, `order2`: internal-then-external — this is the
  counterbalancing trick, covered in its own section below).

We load the **real** upstream files here (not fake data) and just use 2 of the 144 trials, so
everything you see is genuinely the production content, only truncated.


In [ ]:
with open(PREPARE_DIR / "task_rm_v1.json") as f:
    taskbase_2op = json.load(f)

with open(PREPARE_DIR / "rm_task_2op_inst.json") as f:
    instr_2op = json.load(f)

# Just the first 2 of the 144 stimuli: one "imagined" trial, one "perceived" trial.
demo_trials = taskbase_2op["task1"]["trials"][:2]
demo_trials


In [ ]:
task_item = psy.get_task_template()

task_item["taskname"] = "demo_rm_2op"
task_item["chain_type"] = "item"                       # <-- the fork: one stimulus, one response
task_item["contexts"] = ["perceived", "imagined"]
task_item["contexts_id"] = ["perceived", "imagined"]
task_item["context_present"] = False
task_item["parser"] = "ResponseRmStEI"                 # combined Word_2+Rating+Judgment+Confidence, external-first
task_item["instructions"] = instr_2op["order1"]         # the real instruction text, external-before-internal order

task_item["items"] = {}
for i, trial in enumerate(demo_trials):
    ttype = trial["source"].split(" ")[-1].lower()      # "Second Word Imagined" -> "imagined"
    trcode = f"{ttype}_{i + 1}"
    task_item["items"][trcode] = [
        {
            "stimulus": {
                "Word_Pair": {
                    "word_1": trial["stim"]["word 1"],
                    "word_2": trial["stim"]["word 2"],
                },
            },
            "trcode": trcode,
        }
    ]

print(json.dumps(task_item, indent=2)[:1500], "\n... (truncated)")


**`ResponseRmStEI`** (defined in `psychscanner.datasets.prompts.parser`) is the pydantic
schema the model's single response must match for this shape:

```python
class ResponseRmStEI(BaseModel):
    Word_2: str
    Rating: float                              # 0-100 relatedness
    Judgment: Literal["external", "internal"]
    Confidence: Literal[1, 2, 3, 4, 5, 6]
```

One response, four fields — matching the four components described in the instructions.

**Check**: does this match the real `rm_2op.json`? We reused the exact same `instr_2op["order1"]`
object and the exact same first two stimuli, so this should be an exact match, not just "close enough".


In [ ]:
with open(RM_TASKS_DIR / "rm_2op.json") as f:
    real_rm_2op = json.load(f)

assert task_item["instructions"] == real_rm_2op["instructions"], "instructions block should match verbatim"
assert task_item["items"]["imagined_1"] == real_rm_2op["items"]["imagined_1"], "first trial should match verbatim"
assert task_item["items"]["perceived_2"] == real_rm_2op["items"]["perceived_2"], "second trial should match verbatim"
print("Recipe 1 matches the real rm_2op.json content exactly (on the trials/fields compared).")


**Does `psychscanner` actually accept this as a valid task?** Rather than just trusting the
JSON shape, let's write it to disk and hand it to `psychscanner` for real, the same way the
production runner scripts (`simulation/run_rm_task_single_turn_trial_chain.py`, etc.) do —
`ExpCardInit` -> `ExpCard`. This step validates and loads the file; it does **not** run a
simulation (that needs a live LLM — Ollama in production, see `simulation/README.md`).


In [ ]:
task_item_path = SCRATCH_DIR / "demo_rm_2op.json"
with open(task_item_path, "w") as f:
    json.dump(task_item, f, indent=2)

card_in = psy.ExpCardInit()
card_in.proj_dir = SCRATCH_DIR
card_in.projectname = "DEMO"
card_in.task_file = task_item_path
card_in.cogtype = "no"           # skip persona-file loading -- not needed for this walkthrough
card_in.parser = "dynamic"       # resolve the parser class from the task JSON's own "parser" field, like the real runners do

expcard = psy.ExpCard(card_in)
print("Loaded taskname:", expcard.task_data["taskname"])
print("Loaded chain_type:", expcard.task_data["chain_type"])
print("Simulation output would go to:", expcard.data_root_dir)


## Recipe 2: Single-turn, trial chain (`chain_type="trial"`)

**Shape:** same underlying stimulus and same four questions as Recipe 1, but instead of one
combined response, the trial is broken into **four sequential steps** — the model answers one
sub-question at a time, each with its own instructions and its own response schema. The
production version is `rm_2op_tc.json` ("tc" = trial chain).

Each step in a trial reuses one instruction block from the *same* `instr_2op["order1"]` dict
used in Recipe 1 — nothing new to load.


In [ ]:
task_trial = psy.get_task_template()

task_trial["taskname"] = "demo_rm_2op_tc"
task_trial["chain_type"] = "trial"              # <-- the fork: one trial, several sequential steps
task_trial["contexts"] = ["perceived", "imagined"]
task_trial["contexts_id"] = ["perceived", "imagined"]
task_trial["context_present"] = False
task_trial["parser"] = "AllResponseRMEI"        # per-step parser union: Word2 | RelatednessRating | JudgmentEI | Confidence16
task_trial["instructions"] = {"definition": instr_2op["order1"]["definition"]}

task_trial["items"] = {}
for i, trial in enumerate(demo_trials):
    ttype = trial["source"].split(" ")[-1].lower()
    trcode = f"{ttype}_{i + 1}"

    task_trial["items"][trcode] = [
        {   # step 1: report/imagine word_2
            "stimulus": {
                "instructions": {"WORD_PAIR_TASK": instr_2op["order1"]["INSTRUCTIONS"]["WORD_PAIR_TASK"]},
                "Word_Pair": {"word_1": trial["stim"]["word 1"], "word_2": trial["stim"]["word 2"]},
                "Query": "Please report the **Word_2** as per instructions. Word 2 = ",
            },
            "trcode": trcode,
        },
        {   # step 2: relatedness rating
            "stimulus": {
                "instructions": {
                    "RELATEDNESS_RATING_BETWEEN_WORD_1_AND_WORD_2":
                        instr_2op["order1"]["INSTRUCTIONS"]["RELATEDNESS_RATING_BETWEEN_WORD_1_AND_WORD_2"]
                },
                "Query": "Please report **Relatedness Rating** between the two words in the earlier word-pair as per instructions. Relatedness Rating = ",
            },
            "trcode": trcode,
        },
        {   # step 3: internal/external judgment
            "stimulus": {
                "instructions": {
                    "JUDGE_GENERATION_TYPE_OF_WORD_2":
                        instr_2op["order1"]["INSTRUCTIONS"]["JUDGE_GENERATION_TYPE_OF_WORD_2"]
                },
                "Query": "Please select and report the **Judgment** of the word 2 value regarding genration type using the earlier word-pair. Select one of the generation type options as per the instructions. Judgment = ",
            },
            "trcode": trcode,
        },
        {   # step 4: confidence rating
            "stimulus": {
                "instructions": {
                    "CONFIDENCE_ON_JUDGEMENT_OF_GENERATION_TYPE":
                        instr_2op["order1"]["INSTRUCTIONS"]["CONFIDENCE_ON_JUDGEMENT_OF_GENERATION_TYPE"]
                },
                "Query": "Please report the **Confidence** on your judgment being **correct** as per the confidence rating scale insturction in your generation type judgment ealier. Confidence = ",
            },
            "trcode": trcode,
        },
    ]

print("Steps in the first trial:", len(task_trial["items"]["imagined_1"]))


Notice the shape difference from Recipe 1: `items["imagined_1"]` is now a **list of 4 dicts**
(one per step) instead of a list of 1. Each step only carries the instructions relevant to
*that* step — the model never sees all four instruction blocks at once.

**`AllResponseRMEI`** is a `Union` of four small per-step schemas (`Word2`, `RelatednessRating`,
`JudgmentEI`, `Confidence16`) — whichever one matches the current step's expected answer type.

**Check** against the real `rm_2op_tc.json`:


In [ ]:
with open(RM_TASKS_DIR / "rm_2op_tc.json") as f:
    real_rm_2op_tc = json.load(f)

assert task_trial["items"]["imagined_1"] == real_rm_2op_tc["items"]["imagined_1"]
assert task_trial["items"]["perceived_2"] == real_rm_2op_tc["items"]["perceived_2"]
print("Recipe 2 matches the real rm_2op_tc.json content exactly (on the trials compared).")


In [ ]:
task_trial_path = SCRATCH_DIR / "demo_rm_2op_tc.json"
with open(task_trial_path, "w") as f:
    json.dump(task_trial, f, indent=2)

card_in = psy.ExpCardInit()
card_in.proj_dir = SCRATCH_DIR
card_in.projectname = "DEMO"
card_in.task_file = task_trial_path
card_in.cogtype = "no"
card_in.parser = "dynamic"

expcard = psy.ExpCard(card_in)
print("Loaded chain_type:", expcard.task_data["chain_type"])
print("Simulation output would go to:", expcard.data_root_dir)


## Recipe 3: Episodic conversation, no feedback (`chain_type="task"`)

**Shape:** this is a genuinely different structure, not just a rearrangement of Recipe 1/2.
The model first goes through an "encoding" phase — shown a handful of word-pairs one at a time,
in a single ongoing conversation (`memory="Convo"`, so it remembers earlier turns). Later, in
the same conversation, it's "tested": shown just the first word of a pair it saw earlier and
asked to recall/judge the second word — this is what a `"test:imagined_1"`-style `trcode`
means. This shape has its own, independently-written instruction text — it does **not** reuse
`instr_2op` from Recipes 1/2 (a subtly different wording of the word-pair instructions, for
instance). Reproduced below verbatim from `make_rm_tasks_exp2_nofb.ipynb`, not paraphrased, so
the equality checks later are a genuine fidelity check rather than a foregone conclusion.

Two structural differences from Recipes 1 & 2 that matter:
- `tasktype` becomes `"episodic_system"` (overriding the template's default `"sc"`).
- Every item carries its own `system_message` — the model doesn't get one static instruction
  block, it gets a system message per-turn, which the encoding items and test items set differently.


In [ ]:
part_1_instructions = {
    "task_definition": [
        'You are a helpful participant performing a task with two different components for successful response.',
        'Each component refers to a unique problem about the task as described in the instructions below.',
        "Two components of the task are: ['WORD_PAIR_TASK', 'RELATEDNESS_RATING_BETWEEN_WORD_1_AND_WORD_2']",
        'Follow all the instructions related to different components of the task to give accurate response.',
        '**COMMITMENT** You have made the commitment to make sure to follow all the the instructions for each of the components and formatting your task response.'
    ],
    "INSTRUCTIONS":{
        'WORD_PAIR_TASK': {
            '**Identify the Word Pair**': "Look for the word pair, referred to as 'word_1' and 'word_2'.",
            "**Check if 'word_2' is Provided**": [
                "If 'word_2' is a complete and valid English word, report that word as it is.",
                "If 'word_2' contains a blank value ('_______'), proceed to the next step."],
            "**If 'word_2' is a blank in the word-pair then imagine**": ["**Replace blank value of 'word_2' by using your an imagination to create **an** english word to complete the word pair.**"],
            'You are **prohibted** to imagine': ["Use of given 'word_1' in any form.",
                                       'Compound words made of smaller word,',
                                       'Symbols and numbers like: _/^(?=.*?[1-9])[0-9()-]+$,',
                                       'anything outside of the words would be not in a english dictionary.',
                                       'Mathmetical or algebraic equations'],
            'You are **allowed** to imagine': 'singular english words that are **not** prohibited as described above.'
        },
        'RELATEDNESS_RATING_BETWEEN_WORD_1_AND_WORD_2': {
            'definition': ["Numeric value in between 0 (not realated at all) and 100 (very highly related) for Relatedness between two words similarity between the values of 'word_1' and 'word_2'.",
            'It can be based on mutiple factors,for example:',
            'similar sounding (phonetics),',
            'meaning (semantics),',
            'or categorical (whether the two words can related through common categories).',
            ''],
            'relatedness_rating_scale': ['Use any value in the range of 0 to 100 in relatedness rating scale.',
                                         '**0** signifies the words are **not at all related**.',
                                         '**100** signifies the words **very highly related**.',
                                         'Intermediate values between 0 and 100 denote intermediate values.','Relateness value is **stricity defined in the **range of 0: not related at all, TO 100: very highly related**.',
                                         '**ALWAYS FOLLOW THESE INSTRUCTION**'],
            'type': 'number'
        },
    }
}

part_2_instructions_order1 = {
    "task_definition": [
        "You are a helpful participant performing a task based on **previously completed word-pair task** with two different components for successful response.",
        "Here, you will be asked to make judgements about the word you reported as **word_2** in the previous task based on the **Test_Word**",
        "Two components of the current task are: ['JUDGE_GENERATION_TYPE_OF_WORD_2', 'CONFIDENCE_ON_JUDGEMENT_OF_GENERATION_TYPE']",
        "Follow all the instructions related to different components of the task to give accurate response.",
        "**COMMITMENT** You have made the commitment to make sure to follow all the the instructions for each of the components and formatting your task response."
    ],
    "INSTRUCTIONS": {
        "JUDGE_GENERATION_TYPE_OF_WORD_2": {
            "definition": [
                "During the word-pair task, **Test_Word** would have appeared as **Word_1** in one of the pair for which you reported the same word as **word_2** or imagined it.",
                "Judge the nature of generation of the value of 'word_2' on following options:"
            ],
            "options": {"enum": ["external", "internal"]},
            "option_descriptions": {
                "external": "**Only If** the 'word_2' was **provided in the task as an english word.**.",
                "internal": "**Only If** the 'word_2' was **imagined by you to complete the word-pair and replace the blank.**"
            }
        },
        "CONFIDENCE_ON_JUDGEMENT_OF_GENERATION_TYPE": {
            "definition": [
                "Report the **confidence level** about the **selected option** for the judgment of generation type of the 'word_2'.",
                "**Confidence level** indexes your ability **to know** the **level of certainty** that your reported **generation judgment is correct**.",
                "In the context of this task, it is your ability to report **level of certainty** in your judgments about the the generation type for the value of 'word_2'.",
                "Use the confidence rating scale faithfully to report the **level of certainty** as your confidence rating in the ordinal confidence scale described below.",
                "The ordinal values of the confidence scale from 1 to 6.",
                "Higher the numberic value of confidence, higher is your confidence level in the judgment about the generation type being correct.",
                "**Follow the instructions and confidence scale be able to able to be able to faithfully report the confidence level in your selected generation type for the value of 'word_2'."
            ],
            "enum": [1, 2, 3, 4, 5, 6],
            "confidence_scale": {
                "**1**": "**Not at all confident.**",
                "**2**": "**Slightly confident.**",
                "**3**": "**Moderately confident.**",
                "**4**": "**Fairly confident.**",
                "**5**": "**Very confident.**",
                "**6**": "**Highly confident.**"
            },
            "type": "number"
        }
    }
}

source_corrans_map = {"imagined": "internal", "perceived": "external"}


In [ ]:
task_convo = psy.get_task_template()
task_convo["taskname"] = "rm_2op_convo"                 # matches real production naming for this shape
task_convo["tasktype"] = "episodic_system"               # <-- overrides the "sc" default
task_convo["chain_type"] = "task"                        # <-- the fork: conversation with memory
task_convo["contexts"] = ["perceived", "imagined", "test:perceived", "test:imagined"]
task_convo["contexts_id"] = ["perceived", "imagined", "test:perceived", "test:imagined"]
task_convo["context_present"] = False
task_convo["parser"] = "TwoResponses"
task_convo["instructions"] = {"definition": ""}          # unused at this level -- instructions live per-item instead

items_encoding = {}
for i, trial in enumerate(demo_trials):
    ttype = trial["source"].split(" ")[-1].lower()
    trcode = f"{ttype}_{i + 1}"
    items_encoding[trcode] = [
        {
            "stimulus": {"Word_Pair": {"word_1": trial["stim"]["word 1"], "word_2": trial["stim"]["word 2"]}},
            "trcode": trcode,
            "system_message": part_1_instructions,       # <-- per-item system message, shape 3/4's signature feature
        }
    ]

items_test = {}
for trcode, item in items_encoding.items():
    items_test["test:" + trcode] = [
        {
            "stimulus": {"Test_Word": item[0]["stimulus"]["Word_Pair"]["word_1"]},
            "trcode": "test:" + trcode,
            "corrAns": source_corrans_map[trcode.split("_")[0]],   # ground truth, used for grading/feedback
            "og_source": item[0]["stimulus"],
            "system_message": part_2_instructions_order1,
        }
    ]

task_convo["items"] = {**items_encoding, **items_test}
list(task_convo["items"].keys())


**`TwoResponses`** is the response schema for this shape — much looser than the earlier ones,
because a single turn only ever answers one of two possible questions (`WORD_PAIR_TASK` during
encoding, or the judgment+confidence pair during test):

```python
class TwoResponses(BaseModel):
    Response_1: str
    Response_2: str
```

**Check** against the real `rm_2op_convo.json` — the encoding item should match exactly (same
stimulus, same `part_1_instructions`); the test item's `corrAns` and `og_source` should match too.


In [ ]:
with open(RM_TASKS_DIR / "rm_2op_convo.json") as f:
    real_rm_2op_convo = json.load(f)

assert task_convo["items"]["imagined_1"] == real_rm_2op_convo["items"]["imagined_1"]
assert task_convo["items"]["test:imagined_1"][0]["corrAns"] == real_rm_2op_convo["items"]["test:imagined_1"][0]["corrAns"]
assert task_convo["items"]["test:imagined_1"][0]["system_message"] == real_rm_2op_convo["items"]["test:imagined_1"][0]["system_message"]
print("Recipe 3 matches the real rm_2op_convo.json content exactly (on the trials compared).")


In [ ]:
task_convo_path = SCRATCH_DIR / "demo_rm_2op_convo.json"
with open(task_convo_path, "w") as f:
    json.dump(task_convo, f, indent=2)

card_in = psy.ExpCardInit()
card_in.proj_dir = SCRATCH_DIR
card_in.projectname = "DEMO"
card_in.task_file = task_convo_path
card_in.cogtype = "no"
card_in.parser = "dynamic"
card_in.memory = "Convo"          # <-- must be set explicitly: this shape needs cross-turn memory

expcard = psy.ExpCard(card_in)
print("Loaded tasktype:", expcard.task_data["tasktype"])
print("Loaded chain_type:", expcard.task_data["chain_type"])
print("Simulation output would go to:", expcard.data_root_dir)


## Recipe 4: Episodic conversation, with feedback

**Shape:** identical JSON structure to Recipe 3 — same `tasktype`, `chain_type`, `parser`,
same encoding/test item shape. The only JSON-level difference is the instruction text
mentioning feedback (`"You will also be provided with feedback..."`). Everything else that
makes this "feedback" happen lives **outside** the JSON, in the `ExpCard` configuration and in
`simulation/rm_msg_injection.py`.


In [ ]:
part_1_instructions_fb = copy.deepcopy(part_1_instructions)
part_1_instructions_fb["task_definition"] += [
    "You will also be provided with feedback on previous trials in the word-pair task.",
    "Try to use the feedback to improve your performance on the task.",
    "Try to be as accurate as possible.",
]

part_2_instructions_fb = copy.deepcopy(part_2_instructions_order1)
part_2_instructions_fb["task_definition"] += [
    "You will also be provided with feedback on previous trials from now on for your response to the test-word.",
    "Try to use the feedback to improve your performance on the task.",
    "Try to be as accurate as possible.",
]

task_convo_fb = copy.deepcopy(task_convo)
task_convo_fb["taskname"] = "rm_2op_convo_fb"

for trcode in list(items_encoding.keys()):
    task_convo_fb["items"][trcode][0]["system_message"] = part_1_instructions_fb
for trcode in list(items_test.keys()):
    task_convo_fb["items"][trcode][0]["system_message"] = part_2_instructions_fb

print("JSON structure is otherwise unchanged from Recipe 3 -- only the instruction text differs.")


The actual feedback mechanism is wired up on the **`ExpCard`**, not the JSON: setting
`card_in.feedback = "1"` and `card_in.feedback_fn` to a callback class. In production
(`simulation/run_rm_task_episodic_fb.py`) that callback is `Stim_Trial_Injection` from
`simulation/rm_msg_injection.py`. After each test-item response, it:

1. Compares the model's parsed `Judgment` against the trial's `corrAns` (which we set above).
2. Builds a feedback string (`"**CORRECT: ...**"` / `"**INCORRECT: ...**"`).
3. Injects that string as the next turn's message, ahead of the next trial's actual stimulus.

That's *why* Recipe 3/4 bothered to set `corrAns` on the test items — it's not used by the
JSON schema at all, it's read directly by `Stim_Trial_Injection` at run time.


In [ ]:
task_convo_fb_path = SCRATCH_DIR / "demo_rm_2op_convo_fb.json"
with open(task_convo_fb_path, "w") as f:
    json.dump(task_convo_fb, f, indent=2)

card_in = psy.ExpCardInit()
card_in.proj_dir = SCRATCH_DIR
card_in.projectname = "DEMO"
card_in.task_file = task_convo_fb_path
card_in.cogtype = "no"
card_in.parser = "dynamic"
card_in.memory = "Convo"
card_in.feedback = "1"            # <-- turn feedback on
# card_in.feedback_fn = Stim_Trial_Injection   # <-- would import from simulation/rm_msg_injection.py in production

expcard = psy.ExpCard(card_in)
print("Loaded taskname:", expcard.task_data["taskname"])
print("feedback flag on the card:", card_in.feedback)


**Check** against the real `rm_2op_convo_feedback_allparts.json`:


In [ ]:
with open(RM_TASKS_DIR / "rm_2op_convo_feedback_allparts.json") as f:
    real_rm_2op_convo_fb = json.load(f)

assert task_convo_fb["items"]["imagined_1"][0]["system_message"] == real_rm_2op_convo_fb["items"]["imagined_1"][0]["system_message"]
assert task_convo_fb["items"]["test:imagined_1"][0]["system_message"] == real_rm_2op_convo_fb["items"]["test:imagined_1"][0]["system_message"]
print("Recipe 4 matches the real rm_2op_convo_feedback_allparts.json content exactly (on the trials compared).")


## Bonus: the `_r` counterbalancing trick

Every shape above has an `_r` sibling (`rm_2op_r.json`, `rm_2op_tc_r.json`,
`rm_2op_convo_r.json`, `rm_2op_convo_feedback_allparts_r.json`). The point of an `_r` file
is **counterbalancing**: the judgment options are presented in the opposite order
(`internal` before `external`, instead of `external` before `internal`), so that across many
simulated participants, option order itself can't be the thing driving a systematic answer
bias — half the runs see one order, half see the other.

The trick is always the same one swap: `instr_2op["order1"]` -> `instr_2op["order2"]` for the
`JUDGE_GENERATION_TYPE_OF_WORD_2` instruction block. Everything else about the task
(stimuli, other instructions, parser type, chain_type) stays identical. Demonstrated here on
Recipe 1's task, but the same one-line swap applies to all four shapes:


In [ ]:
task_item_r = copy.deepcopy(task_item)
task_item_r["taskname"] = "demo_rm_2op_r"
task_item_r["instructions"]["INSTRUCTIONS"]["JUDGE_GENERATION_TYPE_OF_WORD_2"] = instr_2op["order2"]["JUDGE_GENERATION_TYPE_OF_WORD_2"]
task_item_r["parser"] = "ResponseRmStIE"   # the parser's option Literal order flips too: IE instead of EI

print("Recipe 1 enum order:  ", task_item["instructions"]["INSTRUCTIONS"]["JUDGE_GENERATION_TYPE_OF_WORD_2"]["options"]["enum"])
print("Recipe 1_r enum order:", task_item_r["instructions"]["INSTRUCTIONS"]["JUDGE_GENERATION_TYPE_OF_WORD_2"]["options"]["enum"])

with open(RM_TASKS_DIR / "rm_2op_r.json") as f:
    real_rm_2op_r = json.load(f)
assert task_item_r["instructions"] == real_rm_2op_r["instructions"]
print("Matches the real rm_2op_r.json instructions exactly.")


> **A bug this project actually had here:** the production notebooks that build shapes 3/4
> (`make_rm_tasks_exp2_nofb.ipynb`, `make_rm_tasks_exp2_fb.ipynb`) used to build the base and
> `_r` item dicts by calling a helper *twice on the same shared dict* to attach `order1` then
> `order2` — since the helper mutated its input in place instead of copying, the second call
> silently overwrote the first everywhere, so the "base" file also ended up with the `_r`
> instruction order. Fixed by making that helper `copy.deepcopy` its input before mutating it.
> The lesson generalizes beyond this project: **never mutate a shared dict in a "make a
> variant" helper — copy first.**


## Where to go from here

- **The real generator notebooks**: `make_rm_tasks_exp1.ipynb` (shapes 1 & 2),
  `make_rm_tasks_exp2_nofb.ipynb` (shape 3), `make_rm_tasks_exp2_fb.ipynb` (shape 4) — same
  patterns as here, at full scale (144/30/10/5 trials instead of 2), writing to
  `data/raw/gen_task/`.
- **Running a real simulation**: `simulation/run_rm_task_*.py` — same `ExpCardInit` ->
  `ExpCard` -> `ScannerModel` pattern shown here, but pointed at a real Ollama model instead of
  the mock model, and `card_in.nsim = 100` simulated participants instead of the implicit 1
  used when `cogtype="no"`. See `simulation/README.md` for the full pipeline (SLURM job
  scripts, model list files, task list files).
- **`data/raw/rm_tasks/`** is the actual source of truth this whole project's simulation data
  was generated from — everything in this notebook was checked against it, not the other way
  around.

**Full field glossary for `ExpCardInit`** (used across the recipes above):

| Field | Used for |
|---|---|
| `model` / `family` | Which LLM to call, and its provider (`"ollama"` in production; the mock model used here needs no live model). |
| `parameters` | Model call kwargs (e.g. `temperature`). |
| `memory` | `"SingleTurn"` (shapes 1/2, stateless) or `"Convo"` (shapes 3/4, remembers earlier turns). |
| `persona_files` | Persona/population JSONs (skipped here via `cogtype="no"`). |
| `task_file` | Path to the task JSON — everything built above, once written to disk. |
| `tunnel_status` | Checkpointing during long runs — `"0"` here since these are instant demo runs. |
| `projectname` | Top-level folder name for simulation output. |
| `parser` | `"dynamic"` resolves the parser class from the task JSON's own `"parser"` field, same as every real runner script. |
| `cogtype` | `"custom"` loads persona files as simulated participants; `"no"`/`"assistant"` skip that and default to 1 simulated run. |
| `nsim` | How many simulated participants/runs (100 in production). |
| `chain_type` | Can override the task JSON's own `chain_type` if set here; left alone above so the JSON's own value drives it. |
| `feedback` / `feedback_fn` | Turns on the `Stim_Trial_Injection`-style feedback mechanism (shape 4 only). |
